In [ ]:
import pandas as pd
from datetime import *
import os 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks as fp

# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES, LABEL_DF, Feature_DF

SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}
COL_Prefixes ={"HeartRate":"HR_","Metadata":"MD_","WatchAccelerometer":"ACC_", "WatchGravity":"G_","WatchGyroscope":"GYRO_","WatchOrientation":"OR_","WatchTotalAcceleration":"TACC_","WatchMagnetometer":"MAG_"}

RAWDATAFILES={}
MERGE_FILES=1

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
PREPROC_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"250MS_labeled.pkl"))
EXERCISE_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"exercise_dict.pkl"))


exercise_tuple = [tup for tup in EXERCISE_DF.set_index("key").itertuples()]
#print(exercise_tuple[2].value)
EXERCISE_DF

In [ ]:
from scipy.signal import find_peaks, savgol_filter

def auto_find_peaks(signal, sampling_rate=1.0, smooth_window=11, poly_order=3):
    """
    Automatically smooths and detects peaks in a signal.

    Parameters:
        signal (array-like): Input signal data.
        sampling_rate (float): Samples per unit (Hz if time series).
        smooth_window (int): Window length for Savitzky-Golay smoothing (must be odd).
        poly_order (int): Polynomial order for smoothing.

    Returns:
        peaks (ndarray): Indices of detected peaks.
        properties (dict): Peak properties from scipy.signal.find_peaks.
    """
    signal = np.asarray(signal)

    # Validate parameters
    if smooth_window >= len(signal):
        smooth_window = len(signal) - (1 - len(signal) % 2)  # Ensure odd and < len(signal)
    if smooth_window < 3:
        smooth_window = 3

    # Smooth the signal to reduce noise
    smoothed = savgol_filter(signal, smooth_window, poly_order)

    # Auto threshold: use mean + std as a starting point
    height_threshold = np.mean(smoothed) + 0.5 * np.std(smoothed)

    # Auto minimum distance: avoid detecting peaks too close
    min_distance = int(sampling_rate * 0.1)  # e.g., 0.1 seconds apart

    # Detect peaks
    peaks, properties = find_peaks(
        smoothed,
        height=height_threshold,
        distance=min_distance,
        prominence=np.std(smoothed) * 0.3
    )

    return peaks, properties, smoothed



In [ ]:
# Compressed Base Features + Ground Truth
# Mean, Std, fft
feature_cols = PREPROC_DF.columns[:-5]
target_cols = PREPROC_DF.columns[-5:]

# Window: 5S = 5*4*250ms = 20 slices
fs = 4  # Sampling frequency (Hz)
duration = 5  # seconds
window=fs*duration

# Cut off between Exercise and breaks / switches :2.5s (>=50%, >=10 slices), 2nd Iteration:75%
target_threshold=15

#fft threshhold, measure amount of frequencies with amplitude higher than 
fft_thh = 20

base_features=[]
for col in feature_cols:
    base_features.append("mean_"+col)
    base_features.append("std_"+col)
    base_features.append("fft_"+col)
    #base_features.append("n_peak_"+col) not informative
    #base_features.append("n_val_"+col) not informative
    base_features.append("mean_peak_"+col)
    base_features.append("mean_val_"+col)
    
Feature_DF=pd.DataFrame(columns=base_features + ["target","target_name","datetime"],index=range(0,int(abs((PREPROC_DF.shape[0]-window)/window))))

n_row =0

#overlapping windows (step size: window/2)
for i in range(0,PREPROC_DF.shape[0]-window*2, round(window/2)):
    for feature in feature_cols:
        Feature_DF.loc[n_row,"mean_"+feature]= PREPROC_DF.iloc[i:i+window][feature].mean()
        Feature_DF.loc[n_row,"std_"+feature]= PREPROC_DF.iloc[i:i+window][feature].std()
        #fft amount of high amplitude frequencies
        signal = PREPROC_DF.iloc[i:i+window][feature]
        fft_result = np.fft.fft(signal) 
        Feature_DF.loc[n_row,"fft_"+feature]= len([ x for x in abs(fft_result) if x>=fft_thh]) 
        
        #peak & valley
        y_valleys = PREPROC_DF.iloc[i:i+window][feature].dropna().values *-1 
        y = PREPROC_DF.iloc[i:i+window][feature].dropna().values 
        peaks, props, smoothed = auto_find_peaks(y, sampling_rate=250/ (6*np.pi))
        peaks_v, props_v, smoothed_v = auto_find_peaks(y_valleys, sampling_rate=250/ (6*np.pi))
        
        #Feature_DF.loc[n_row,"n_peak_"+feature] = len(peaks)
        #Feature_DF.loc[n_row,"n_val_"+feature] = len(peaks_v)

        Feature_DF.loc[n_row,"mean_peak_"+feature] = smoothed.mean()
        Feature_DF.loc[n_row,"mean_val_"+feature] = smoothed_v.mean()

        #target
        if sum(list(PREPROC_DF.iloc[i:i+window]["target"].notna()))>=target_threshold:
            target_list=list(PREPROC_DF.iloc[i:i+window]["target"])
            target_list.sort(reverse=True)
            target= target_list[0]
            Feature_DF.loc[n_row,"target"] = int(target)
            if ~np.isnan(target):
                Feature_DF.loc[n_row,"target_name"] = exercise_tuple[int(target)].value
        
        #timestamp
        Feature_DF.loc[n_row,"datetime"] = PREPROC_DF.index[i]
    n_row +=1


In [ ]:
#Validation
# print(list(PREPROC_DF.iloc[0:19]["target"].notna()))
# print(first_non_nan([2,np.nan,3,np.nan]))
#print(PREPROC_DF.loc["2025-12-22 11:38:58.250":"2025-12-22 11:39:05.250"])
#Feature_DF["datetime"] = pd.to_datetime(Feature_DF["datetime"])

#Feature_DF[(Feature_DF["datetime"]  >= pd.to_datetime("2025-12-19 15:12:47.750")) & (Feature_DF["datetime"]  <= pd.to_datetime("2025-12-19 15:22:06.750"))]
Feature_DF.sample(50)
Feature_DF.shape

In [ ]:
#Check Correlation
def showCorrelation(cols):
    sns.pairplot(Feature_DF[cols].sample(200), height=2.5)
    plt.tight_layout()

#[x for x in base_features if x.find("mean")>-1]
showCorrelation( list(set([Feature_DF.columns[np.random.randint(1, Feature_DF.shape[1])] for x in range(0,10) ])) )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def testFFT():
    # Generating a sample musical note signal
    fs = 4  # Sampling frequency (Hz)
    duration = 5  # seconds
    #positives
    #signal = PREPROC_DF.loc["2025-12-27 16:52:00":"2025-12-27 16:52:5"]["G_x"]
    #signal = PREPROC_DF.loc["2025-12-27 16:26:27":"2025-12-27 16:26:32"]["G_x"]

    #negatives
    #signal = PREPROC_DF.loc["2025-12-27 16:51:21":"2025-12-27 16:51:26"]["G_x"]
    signal = PREPROC_DF.loc["2025-12-27 16:25:27":"2025-12-27 16:25:32"]["G_x"]


    plt.plot(signal)
    plt.show()

    # Applying FFT
    fft_result = np.fft.fft(signal)
    freq = np.fft.fftfreq(24, d=0.05/fs)
    plt.plot(fft_result)
    plt.show()

    print( len([ x for x in np.abs(fft_result[1:]) if x>=20]) )

    # Plotting the spectrum
    plt.plot(freq[1:], fft_result[1:])
    plt.title('FFT')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Amplitude')
    plt.show()
    print()
testFFT()

In [ ]:
#Check Amount of Sample per Class
df_count = Feature_DF.groupby("target").agg({"target":'count'}).rename(columns={'target': 'target_count'}).sort_values("target_count")
df_count

In [ ]:
#calculate mean for sample threshhold
print(df_count[((df_count.target_count>40) & (df_count.index>1))]["target_count"].mean())

#classes with most samples for baseline
for x in list(df_count.index)[-8: ]:
    print(f"{x},")

In [ ]:
#EXERCISE_DF = pd.concat(EXERCISE_DF,df_count,axis=1)
df_count.sort_index(inplace=True)
EXERCISE_DF=pd.concat([EXERCISE_DF, df_count],axis=1)
EXERCISE_DF


In [ ]:
#Feature_DF.target.value_counts(dropna=False)
Feature_DF

In [ ]:
Feature_DF.to_pickle(os.path.join(PREDICTIONS_Data_DIR,"features.pkl"))
EXERCISE_DF.to_pickle(os.path.join(PREDICTIONS_Data_DIR,"exercise_dict.pkl"))

In [ ]:
# run lines in terminal to convert into html
print(f"cas_gpu_venv\\Scripts\\activate.bat")
print(f"jupyter-nbconvert --to html \"0_Module\\M3_Machine_Learning\\Project\\notebooks\\{os.path.basename(globals()['__vsc_ipynb_file__'])}\"")